# 🎬 Video Transcription & Speaker Diarization

This notebook transcribes an MP4 video file with **speaker diarization** (who said what).

| Step | Tool |
|------|------|
| Audio extraction | `moviepy` + `ffmpeg` |
| Speaker diarization | `pyannote/speaker-diarization-3.1` |
| Transcription | caspi (`base` by default) |
| Output | `.json` + `.pdf` (named after the video) |

### Prerequisites
- A **HuggingFace token** with access to `pyannote/speaker-diarization-3.1`  
  → Accept the licence at https://huggingface.co/pyannote/speaker-diarization-3.1  
  → Create a token at https://huggingface.co/settings/tokens

## 1 · Install everything, then restart runtime

In [ ]:
# ============================================================
# CELL 1 — INSTALL LIBRARIES
# Run this cell first.
# After it finishes, the runtime will restart automatically.
# Then continue from Cell 2.
# ============================================================

# Keep numpy < 2 because pyannote can have issues with numpy 2.x
!pip install -q "numpy<2.0" --force-reinstall

# Main libraries:
# qwen-asr       -> for Caspi ASR
# pyannote.audio -> for speaker diarization
# moviepy        -> to extract audio from MP4
# reportlab      -> to create PDF
# python-bidi    -> helps display Hebrew correctly in PDF
!pip install -q qwen-asr "pyannote.audio>=3.1" torch torchaudio moviepy reportlab python-bidi soundfile librosa accelerate

# System tools and fonts
!apt-get -y install ffmpeg fonts-dejavu

# Hard restart so numpy version is applied correctly
import os
os.kill(os.getpid(), 9)

## 2 · Configuration




In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
from google.colab import userdata

MP4_PATH   = r"C:\Users\Ta7boosh\Downloads\hebrewDATa youtube\piece_of_hebrew.mp4"
OUTPUT_DIR = r"C:\Users\Ta7boosh\Downloads\hebrewDATa youtube"

HUGGINGFACE_TOKEN  = userdata.get('HF_TOKEN')

# WHISPER_MODEL      = "base"
MAX_CHUNK_DURATION = 30.0
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ============================================================
# CELL 2 — USER CONFIGURATION
# Put your video path and output folder here.
# ============================================================

from google.colab import drive, userdata
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Input video path
MP4_PATH = "/content/drive/MyDrive/hebrew/piece_of_hebrew.mp4"

# Output folder
OUTPUT_DIR = "/content/drive/MyDrive/output/"

# Hugging Face token for pyannote
# Add it in Colab Secrets with the name HF_TOKEN
HUGGINGFACE_TOKEN = userdata.get("HF_TOKEN")

# Caspi model instead of Whisper
CASPI_MODEL = "OzLabs/Caspi-1.7B"

# Maximum duration of each diarized chunk
MAX_CHUNK_DURATION = 30.0

# Language for Caspi
# For Hebrew use "Hebrew"
CASPI_LANGUAGE = "Hebrew"

# Check paths
mp4_path = Path(MP4_PATH)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

assert mp4_path.exists(), f"MP4 not found: {mp4_path}"
assert mp4_path.suffix.lower() == ".mp4", "Input must be an .mp4 file"
assert HUGGINGFACE_TOKEN is not None, "HF_TOKEN not found in Colab Secrets"

video_stem = mp4_path.stem

audio_out = output_dir / f"{video_stem}_audio.wav"
json_out  = output_dir / f"{video_stem}_caspi_transcript.json"
pdf_out   = output_dir / f"{video_stem}_caspi_transcript.pdf"

print("Video path :", mp4_path)
print("Audio out  :", audio_out)
print("JSON out   :", json_out)
print("PDF out    :", pdf_out)

## 3 · Imports & setup

In [ ]:
# ============================================================
# CELL 3 — IMPORTS
# ============================================================

import os
import json
import math
import tempfile
from pathlib import Path
from xml.sax.saxutils import escape

import torch

try:
    from moviepy.editor import VideoFileClip
except Exception:
    from moviepy import VideoFileClip

from pyannote.audio import Pipeline
from qwen_asr import Qwen3ASRModel

from bidi.algorithm import get_display

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

DIARIZATION_MODEL = "pyannote/speaker-diarization-3.1"

print("Imports loaded successfully")
print("CUDA available:", torch.cuda.is_available())

## 4 · Helper functions

In [ ]:
# ============================================================
# CELL 4 — HELPER FUNCTIONS
# MP4 -> WAV
# Diarization
# Chunk creation
# Caspi transcription
# JSON/PDF saving
# ============================================================


def extract_audio(video_path: str, audio_path: str) -> None:
    """
    Extract mono 16 kHz WAV audio from an MP4 file.
    Caspi and pyannote work better with clean mono audio.
    """
    clip = VideoFileClip(video_path)
    clip.audio.write_audiofile(
        audio_path,
        fps=16000,
        nbytes=2,
        codec="pcm_s16le",
        ffmpeg_params=["-ac", "1"],
        logger=None,
    )
    clip.close()
    print(f"Audio extracted -> {audio_path}")


def load_caspi_model():
    """
    Load Caspi ASR model.
    Caspi is based on Qwen3-ASR and is loaded using qwen_asr.
    """
    if torch.cuda.is_available():
        dtype = torch.bfloat16
        device_map = "cuda:0"
        batch_size = 4
    else:
        dtype = torch.float32
        device_map = "cpu"
        batch_size = 1

    print("Loading Caspi model...")
    print("Device:", device_map)
    print("dtype :", dtype)

    model = Qwen3ASRModel.from_pretrained(
        CASPI_MODEL,
        dtype=dtype,
        device_map=device_map,
        max_inference_batch_size=batch_size,
        max_new_tokens=256,
    )

    print("Caspi model loaded")
    return model


def run_diarization(audio_path: str, hf_token: str):
    """
    Run pyannote speaker diarization.
    Output tells us: who spoke when.
    """
    print("Loading diarization model...")

    pipeline = Pipeline.from_pretrained(
        DIARIZATION_MODEL,
        token=hf_token,
    )

    if torch.cuda.is_available():
        pipeline.to(torch.device("cuda"))

    print("Running diarization...")
    diarization = pipeline(audio_path)

    print("Diarization finished")
    return diarization


def get_annotation_from_diarization(diarization):
    """
    pyannote output can be different depending on version.

    Old format:
        diarization.itertracks(...)

    New format:
        diarization.speaker_diarization.itertracks(...)

    This function supports both.
    """
    if hasattr(diarization, "itertracks"):
        return diarization

    if hasattr(diarization, "speaker_diarization"):
        return diarization.speaker_diarization

    if hasattr(diarization, "exclusive_speaker_diarization"):
        return diarization.exclusive_speaker_diarization

    raise TypeError("Unsupported diarization output format")


def diarization_to_chunks(diarization, max_duration: float = MAX_CHUNK_DURATION):
    """
    Convert diarization output into chunks.

    Rules:
    1. Get speaker turns.
    2. Merge consecutive turns from the same speaker.
    3. Split long chunks into smaller chunks.
    """
    annotation = get_annotation_from_diarization(diarization)

    raw_turns = [
        {
            "start": float(segment.start),
            "end": float(segment.end),
            "speaker": speaker,
        }
        for segment, _, speaker in annotation.itertracks(yield_label=True)
    ]

    if not raw_turns:
        return []

    # Sort by start time
    raw_turns = sorted(raw_turns, key=lambda x: x["start"])

    # Merge consecutive turns from the same speaker
    merged = [raw_turns[0].copy()]

    for turn in raw_turns[1:]:
        same_speaker = turn["speaker"] == merged[-1]["speaker"]

        if same_speaker:
            merged[-1]["end"] = max(merged[-1]["end"], turn["end"])
        else:
            merged.append(turn.copy())

    # Split chunks longer than max_duration
    chunks = []

    for turn in merged:
        start = turn["start"]
        end = turn["end"]
        speaker = turn["speaker"]

        while start < end:
            chunk_end = min(start + max_duration, end)

            if chunk_end - start > 0.5:
                chunks.append({
                    "start": round(start, 3),
                    "end": round(chunk_end, 3),
                    "speaker": speaker,
                })

            start = chunk_end

    return chunks


def extract_text_from_caspi_result(result):
    """
    Caspi result may be:
    - a list of objects with .text
    - a dict
    - a string

    This function safely extracts the text.
    """
    if result is None:
        return ""

    if isinstance(result, list):
        if len(result) == 0:
            return ""
        item = result[0]
    else:
        item = result

    if hasattr(item, "text"):
        return item.text.strip()

    if isinstance(item, dict):
        return str(item.get("text", "")).strip()

    if isinstance(item, str):
        return item.strip()

    return str(item).strip()


def transcribe_chunks_with_caspi(audio_path: str, chunks: list, model) -> list:
    """
    Run Caspi on every diarized chunk.
    This replaces the old Whisper transcription function.
    """
    results = []

    with tempfile.TemporaryDirectory() as tmp_dir:
        for idx, chunk in enumerate(chunks):
            start = chunk["start"]
            end = chunk["end"]
            duration = round(end - start, 3)

            chunk_path = os.path.join(tmp_dir, f"chunk_{idx:04d}.wav")

            # Cut chunk from full audio using ffmpeg
            os.system(
                f'ffmpeg -y -loglevel quiet '
                f'-i "{audio_path}" '
                f'-ss {start:.3f} -to {end:.3f} '
                f'-ar 16000 -ac 1 "{chunk_path}"'
            )

            # Caspi transcription
            caspi_result = model.transcribe(
                audio=chunk_path,
                language=CASPI_LANGUAGE,
            )

            text = extract_text_from_caspi_result(caspi_result)

            results.append({
                "id": idx + 1,
                "start": round(start, 3),
                "end": round(end, 3),
                "duration": duration,
                "speaker": chunk["speaker"],
                "transcription": text,
            })

            if (idx + 1) % 5 == 0 or (idx + 1) == len(chunks):
                print(f"Transcribed {idx + 1}/{len(chunks)} chunks")

    return results


def save_json(transcriptions: list, path: str) -> None:
    """
    Save transcription results to JSON.
    JSON keeps speaker, time, duration, and text.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(transcriptions, f, ensure_ascii=False, indent=2)

    print(f"JSON saved -> {path}")


def prepare_pdf_text(text: str) -> str:
    """
    Prepare Hebrew/RTL text for PDF.
    """
    if text is None or str(text).strip() == "":
        return "&lt;silence&gt;"

    text = str(text)
    text = get_display(text)
    text = escape(text)
    return text


def save_pdf(transcriptions: list, path: str, title: str) -> None:
    """
    Save a clean PDF transcript.
    The PDF shows only speaker + transcription.
    It does not show id/start/end/duration.
    """
    font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
    bold_font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"

    pdfmetrics.registerFont(TTFont("DejaVuSans", font_path))
    pdfmetrics.registerFont(TTFont("DejaVuSans-Bold", bold_font_path))

    doc = SimpleDocTemplate(
        path,
        pagesize=letter,
        leftMargin=2 * cm,
        rightMargin=2 * cm,
        topMargin=2 * cm,
        bottomMargin=2 * cm,
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "DocTitle",
        parent=styles["Heading1"],
        fontName="DejaVuSans-Bold",
        fontSize=16,
        spaceAfter=16,
        textColor=colors.HexColor("#1a1a2e"),
    )

    speaker_style = ParagraphStyle(
        "Speaker",
        parent=styles["Normal"],
        fontName="DejaVuSans-Bold",
        fontSize=10,
        leading=15,
        spaceBefore=10,
        spaceAfter=2,
        textColor=colors.HexColor("#0f3460"),
    )

    text_style = ParagraphStyle(
        "Body",
        parent=styles["Normal"],
        fontName="DejaVuSans",
        fontSize=10,
        leading=15,
        spaceAfter=6,
        textColor=colors.HexColor("#333333"),
        alignment=2,  # right alignment for Hebrew
    )

    story = [
        Paragraph(f"Transcription — {escape(title)}", title_style),
        Spacer(1, 0.3 * cm),
    ]

    for chunk in transcriptions:
        speaker = escape(str(chunk["speaker"]))
        text = prepare_pdf_text(chunk["transcription"])

        story.append(Paragraph(f"{speaker}:", speaker_style))
        story.append(Paragraph(text, text_style))

    doc.build(story)
    print(f"PDF saved -> {path}")


print("Helper functions ready")

## 5 · Step 1 — Extract audio

In [ ]:
# ============================================================
# CELL 5.1 — EXTRACT AUDIO FROM MP4
# ============================================================

# We keep the audio in a temporary file for the whole session
_tmp_dir  = tempfile.mkdtemp()
AUDIO_PATH = os.path.join(_tmp_dir, "audio.wav")

print(f"Extracting audio from '{mp4_path.name}' …")
extract_audio(str(mp4_path), AUDIO_PATH)
print("Done.")

##Cell 5.2 — Run diarization

In [ ]:
print(f"Running speaker diarization with '{DIARIZATION_MODEL}' …")
diarization = run_diarization(AUDIO_PATH, HUGGINGFACE_TOKEN)
print("Diarization complete.")
print(diarization)

## Cell 5.3 — Create chunks from diarization

In [ ]:
# ============================================================
# CELL 5.3 — CONVERT DIARIZATION TO CHUNKS
# Each chunk has:
# start time, end time, speaker label
# ============================================================

MAX_CHUNK_DURATION = 30.0

def diarization_to_chunks(diarization, max_duration: float = MAX_CHUNK_DURATION):
    """
    Convert diarization output into speaker chunks.
    Works with new pyannote DiarizeOutput.
    """

    # New pyannote output sometimes stores the real annotation here
    if hasattr(diarization, "speaker_diarization"):
        annotation = diarization.speaker_diarization
    else:
        annotation = diarization

    raw_turns = []

    # Extract speaker turns
    for seg, _, spk in annotation.itertracks(yield_label=True):
        raw_turns.append({
            "start": float(seg.start),
            "end": float(seg.end),
            "speaker": spk
        })

    if not raw_turns:
        return []

    # Merge consecutive segments from the same speaker
    merged = [raw_turns[0].copy()]

    for turn in raw_turns[1:]:
        if turn["speaker"] == merged[-1]["speaker"]:
            merged[-1]["end"] = turn["end"]
        else:
            merged.append(turn.copy())

    # Split long speaker turns into max 30-second chunks
    chunks = []

    for turn in merged:
        start = turn["start"]
        final_end = turn["end"]

        while start < final_end:
            end = min(start + max_duration, final_end)

            chunks.append({
                "start": start,
                "end": end,
                "speaker": turn["speaker"]
            })

            start = end

    return chunks


print("Updated diarization_to_chunks function loaded ✓")

In [ ]:
print(f"Building chunks (max {MAX_CHUNK_DURATION}s) …")
chunks = diarization_to_chunks(diarization, max_duration=MAX_CHUNK_DURATION)

assert chunks, "ERROR: Diarization returned no speaker turns."
print(f"{len(chunks)} chunk(s) created.\n")

# Preview first 5 chunks
print(f"{'#':>4}  {'Speaker':<14}  {'Start':>7}  {'End':>7}  {'Dur':>6}")
print("-" * 46)
for i, c in enumerate(chunks[:5]):
    dur = c['end'] - c['start']
    print(f"{i+1:>4}  {c['speaker']:<14}  {c['start']:>7.2f}  {c['end']:>7.2f}  {dur:>6.2f}s")
if len(chunks) > 5:
    print(f"  … and {len(chunks) - 5} more")

## Cell 5.4 — Build chunks

In [ ]:
# ============================================================
# CELL 5.4 — BUILD CHUNKS
# ============================================================

print(f"Building chunks (max {MAX_CHUNK_DURATION}s) …")

chunks = diarization_to_chunks(
    diarization,
    max_duration=MAX_CHUNK_DURATION
)

assert chunks, "ERROR: Diarization returned no speaker turns."

print(f"{len(chunks)} chunk(s) created.\n")

# Preview first 5 chunks
print(f"{'#':>4}  {'Speaker':<14}  {'Start':>7}  {'End':>7}  {'Dur':>6}")
print("-" * 46)

for i, c in enumerate(chunks[:5]):
    dur = c["end"] - c["start"]
    print(f"{i+1:>4}  {c['speaker']:<14}  {c['start']:>7.2f}  {c['end']:>7.2f}  {dur:>6.2f}s")

if len(chunks) > 5:
    print(f"  … and {len(chunks) - 5} more")

## Cell 5.5 — Load Caspi

In [ ]:
# ============================================================
# CELL 5.5 — LOAD CASPI MODEL
# This replaces:
# whisper_model = whisper.load_model(WHISPER_MODEL)
# ============================================================

!pip -q install qwen-asr soundfile accelerate

import soundfile as sf
import numpy as np
from qwen_asr import Qwen3ASRModel

CASPI_MODEL = "OzLabs/Caspi-1.7B"

print(f"Loading Caspi model '{CASPI_MODEL}' …")

if torch.cuda.is_available():
    dtype = torch.bfloat16
    device_map = "cuda:0"
    batch_size = 1
else:
    dtype = torch.float32
    device_map = "cpu"
    batch_size = 1

caspi_model = Qwen3ASRModel.from_pretrained(
    CASPI_MODEL,
    dtype=dtype,
    device_map=device_map,
    max_inference_batch_size=batch_size,
    max_new_tokens=256,
)

print("Caspi model loaded ✓")

## 10 · Save outputs

In [ ]:
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from bidi.algorithm import get_display
from reportlab.lib.enums import TA_RIGHT

FONT_REGULAR = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
FONT_BOLD = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"

pdfmetrics.registerFont(TTFont("DejaVuSans", FONT_REGULAR))
pdfmetrics.registerFont(TTFont("DejaVuSans-Bold", FONT_BOLD))


def save_pdf(transcriptions: list, path: str, title: str) -> None:
    """Write a formatted speaker transcript to a PDF with Hebrew support."""

    doc = SimpleDocTemplate(
        path,
        pagesize=letter,
        leftMargin=2*cm,
        rightMargin=2*cm,
        topMargin=2*cm,
        bottomMargin=2*cm,
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "DocTitle",
        parent=styles["Heading1"],
        fontName="DejaVuSans-Bold",
        fontSize=16,
        spaceAfter=16,
        alignment=TA_RIGHT,
    )

    speaker_style = ParagraphStyle(
        "Speaker",
        parent=styles["Normal"],
        fontName="DejaVuSans-Bold",
        fontSize=10,
        leading=15,
        spaceBefore=10,
        spaceAfter=2,
        alignment=TA_RIGHT,
    )

    text_style = ParagraphStyle(
        "Body",
        parent=styles["Normal"],
        fontName="DejaVuSans",
        fontSize=10,
        leading=15,
        spaceAfter=6,
        alignment=TA_RIGHT,
    )

    story = [
        Paragraph(get_display(f"Transcription — {title}"), title_style),
        Spacer(1, 0.3 * cm),
    ]

    for chunk in transcriptions:
        speaker = get_display(f"{chunk['speaker']}:")
        text = chunk["transcription"] or "<silence>"
        text = get_display(text)

        story.append(Paragraph(speaker, speaker_style))
        story.append(Paragraph(text, text_style))

    doc.build(story)
    print(f"✓ PDF saved → {path}")
print("Saving outputs …")
save_json(transcriptions, str(json_out))
save_pdf(transcriptions, str(pdf_out), video_stem)

print(f"\n✅ All done! Files written to: {output_dir}")

## Cell 5.6 — Transcribe chunks with Caspi

In [ ]:
# ============================================================
# CELL 5.6 — TRANSCRIBE CHUNKS WITH CASPI
# This replaces the old Whisper transcribe_chunks call.
# ============================================================

def extract_caspi_text(result):
    """
    Safely extract text from Caspi output.
    Caspi output may be a list, object, dict, or string.
    """

    if result is None:
        return ""

    if isinstance(result, list):
        if len(result) == 0:
            return ""
        result = result[0]

    if hasattr(result, "text"):
        return result.text.strip()

    if isinstance(result, dict):
        return str(result.get("text", "")).strip()

    return str(result).strip()


def transcribe_chunks_caspi(audio_path: str, chunks: list, model) -> list:
    """
    Cut each diarization chunk from the audio,
    then transcribe it with Caspi.
    """

    results = []

    with tempfile.TemporaryDirectory() as tmp_dir:
        for idx, chunk in enumerate(chunks):

            start = chunk["start"]
            end = chunk["end"]
            duration = round(end - start, 3)

            chunk_path = os.path.join(tmp_dir, f"chunk_{idx:04d}.wav")

            # Cut the chunk from the full audio file
            os.system(
                f'ffmpeg -y -loglevel quiet '
                f'-i "{audio_path}" '
                f'-ss {start:.3f} -to {end:.3f} '
                f'-ar 16000 -ac 1 "{chunk_path}"'
            )

            # Read audio as numpy array for Caspi
            audio_array, sr = sf.read(chunk_path)
            audio_array = np.array(audio_array, dtype=np.float32)

            # Caspi transcription
            result = model.transcribe(
                audio=(audio_array, sr),
                language=None
            )

            text = extract_caspi_text(result)

            results.append({
                "id": idx + 1,
                "speaker": chunk["speaker"],
                "start": round(start, 3),
                "end": round(end, 3),
                "duration": duration,
                "transcription": text,
            })

            if (idx + 1) % 5 == 0 or (idx + 1) == len(chunks):
                print(f"Transcribed {idx + 1}/{len(chunks)} chunks …")

    return results


print("Transcribing chunks with Caspi …")

transcriptions = transcribe_chunks_caspi(
    AUDIO_PATH,
    chunks,
    caspi_model
)

print(f"\nTranscription complete — {len(transcriptions)} records.")

## Cell 5.7 — Preview results

In [ ]:
# ============================================================
# CELL 5.7 — PREVIEW TRANSCRIPTION RESULTS
# ============================================================

print(f"{'ID':>4}  {'Speaker':<14}  {'Start':>7}  {'End':>7}  Text")
print("-" * 80)

for t in transcriptions[:10]:
    text_preview = t["transcription"][:50] + ("…" if len(t["transcription"]) > 50 else "")
    print(f"{t['id']:>4}  {t['speaker']:<14}  {t['start']:>7.2f}  {t['end']:>7.2f}  {text_preview}")

if len(transcriptions) > 10:
    print(f"  … and {len(transcriptions) - 10} more records")

## Cell 5.8 — Save JSON and PDF

In [ ]:
# ============================================================
# CELL 5.8 — SAVE OUTPUTS
# Save transcription as JSON and PDF.
# ============================================================

from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from bidi.algorithm import get_display
from reportlab.lib.enums import TA_RIGHT

FONT_REGULAR = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
FONT_BOLD = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"

pdfmetrics.registerFont(TTFont("DejaVuSans", FONT_REGULAR))
pdfmetrics.registerFont(TTFont("DejaVuSans-Bold", FONT_BOLD))


def save_pdf(transcriptions: list, path: str, title: str) -> None:
    """
    Write a formatted speaker transcript to a PDF with Hebrew support.
    """

    doc = SimpleDocTemplate(
        path,
        pagesize=letter,
        leftMargin=2*cm,
        rightMargin=2*cm,
        topMargin=2*cm,
        bottomMargin=2*cm,
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "DocTitle",
        parent=styles["Heading1"],
        fontName="DejaVuSans-Bold",
        fontSize=16,
        spaceAfter=16,
        alignment=TA_RIGHT,
    )

    speaker_style = ParagraphStyle(
        "Speaker",
        parent=styles["Normal"],
        fontName="DejaVuSans-Bold",
        fontSize=10,
        leading=15,
        spaceBefore=10,
        spaceAfter=2,
        alignment=TA_RIGHT,
    )

    text_style = ParagraphStyle(
        "Body",
        parent=styles["Normal"],
        fontName="DejaVuSans",
        fontSize=10,
        leading=15,
        spaceAfter=6,
        alignment=TA_RIGHT,
    )

    story = [
        Paragraph(get_display(f"Transcription — {title}"), title_style),
        Spacer(1, 0.3 * cm),
    ]

    for chunk in transcriptions:
        speaker = get_display(f"{chunk['speaker']}:")
        text = chunk["transcription"] or "<silence>"
        text = get_display(text)

        story.append(Paragraph(speaker, speaker_style))
        story.append(Paragraph(text, text_style))

    doc.build(story)
    print(f"✓ PDF saved → {path}")


print("Saving outputs …")

save_json(transcriptions, str(json_out))
save_pdf(transcriptions, str(pdf_out), video_stem)

print(f"\n✅ All done! Files written to: {output_dir}")

## Cell 5.9 — Remove temporary files

In [ ]:
# ============================================================
# CELL 5.9 — CLEAN TEMPORARY FILES
# ============================================================

import shutil

shutil.rmtree(_tmp_dir, ignore_errors=True)

print("Temporary files removed.")